-- Import nfl python package and load stats to test. ONLY 2026 CURRENTLY

In [9]:
import nflreadpy as nfl

stats = nfl.load_player_stats([2026]).to_pandas()
print(stats.shape)
print("Latest week:", stats["week"].max())
stats.columns.tolist()

(4024, 150)
Latest week: 4


['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'season',
 'week',
 'season_type',
 'game_id',
 'team',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'passing_interceptions',
 'sacks_suffered',
 'sack_yards_lost',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_cpoe',
 'passing_2pt_conversions',
 'pacr',
 'passing_10',
 'passing_16',
 'passing_20',
 'passing_40',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'rushing_10',
 'rushing_12',
 'rushing_20',
 'rushing_40',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_

-- load WR's stats and group all weeks of stats by player to total

In [10]:
wr = stats[(stats["position"] == "WR") & (stats["season_type"] == "REG")]

players = (
    wr.groupby(["player_id", "player_display_name"])
      .agg(
          games=("week", "nunique"),
          targets=("targets", "sum"),
          receptions=("receptions", "sum"),
          yards=("receiving_yards", "sum"),
          tds=("receiving_tds", "sum"),
          air_yards=("receiving_air_yards", "sum"),
          yac=("receiving_yards_after_catch", "sum"),
          epa=("receiving_epa", "sum"),
      )
      .reset_index()
)

players.sort_values("targets", ascending=False).head(15)

,player_id,player_display_name,games,targets,receptions,yards,tds,air_yards,yac,epa
40,00-0036358,CeeDee Lamb,4,46,37,498,4,470,156,41.641878
85,00-0038559,Michael Wilson,4,44,25,258,1,418,68,-6.532638
42,00-0036410,Tee Higgins,4,39,25,401,1,521,96,24.633375
2,00-0031381,Davante Adams,4,38,22,390,2,636,107,14.920643
156,00-0040667,Matthew Golden,4,37,18,274,2,505,106,5.020312
61,00-0037239,Chris Olave,3,36,27,375,1,472,81,27.864391
83,00-0038543,Jaxon Smith-Njigba,3,36,27,405,6,348,121,28.975369
54,00-0036963,Amon-Ra St. Brown,3,35,23,228,5,275,73,10.125158
28,00-0035640,DK Metcalf,4,33,16,213,1,494,54,-0.537352
79,00-0038124,Christian Watson,4,32,20,331,4,327,128,9.975405


-- Set minimum targets for Sample Size
-- Get detailed WR stats 'rates'
-- Check # of WR's left with min targets and check head

In [11]:
MIN_TARGETS = 15  # adjust once you see the data

wr_rates = players[players["targets"] >= MIN_TARGETS].copy()
wr_rates["catch_rate"] = wr_rates["receptions"] / wr_rates["targets"]
wr_rates["yards_per_target"] = wr_rates["yards"] / wr_rates["targets"]
wr_rates["air_yards_per_target"] = wr_rates["air_yards"] / wr_rates["targets"]
wr_rates["yac_per_reception"] = wr_rates["yac"] / wr_rates["receptions"]
wr_rates["epa_per_target"] = wr_rates["epa"] / wr_rates["targets"]
wr_rates["targets_per_game"] = wr_rates["targets"] / wr_rates["games"]

print(wr_rates.shape)
wr_rates.sort_values("epa_per_target", ascending=False).head(15)


(67, 16)


,player_id,player_display_name,games,targets,receptions,yards,tds,air_yards,yac,epa,catch_rate,yards_per_target,air_yards_per_target,yac_per_reception,epa_per_target,targets_per_game
3,00-0031408,Mike Evans,3,16,12,137,2,175,17,22.387218,0.750000,8.562500,10.937500,1.416667,1.399201,5.333333
21,00-0034960,Jakobi Meyers,4,15,13,165,2,149,36,18.263214,0.866667,11.000000,9.933333,2.769231,1.217548,3.750000
129,00-0039915,Ladd McConkey,3,15,12,183,1,208,23,16.918833,0.800000,12.200000,13.866667,1.916667,1.127922,5.000000
128,00-0039901,Keon Coleman,4,16,14,217,1,205,44,17.915049,0.875000,13.562500,12.812500,3.142857,1.119691,4.000000
104,00-0039064,Zay Flowers,3,22,18,352,2,224,165,23.984327,0.818182,16.000000,10.181818,9.166667,1.090197,7.333333
38,00-0036322,Justin Jefferson,3,17,13,179,2,189,52,15.722178,0.764706,10.529412,11.117647,4.000000,0.924834,5.666667
40,00-0036358,CeeDee Lamb,4,46,37,498,4,470,156,41.641878,0.804348,10.826087,10.217391,4.216216,0.905258,11.500000
60,00-0037238,Drake London,3,19,15,274,0,142,152,16.279734,0.789474,14.421053,7.473684,10.133333,0.856828,6.333333
83,00-0038543,Jaxon Smith-Njigba,3,36,27,405,6,348,121,28.975369,0.750000,11.250000,9.666667,4.481481,0.804871,12.000000
45,00-0036554,Nico Collins,2,18,14,193,3,232,46,14.328988,0.777778,10.722222,12.888889,3.285714,0.796055,9.000000


-- Detail WR comparison metrics

In [12]:
features = [
    "targets_per_game",      # how involved he is
    "air_yards_per_target",  # how deep he's used
    "catch_rate",            # reliability
    "yards_per_target",      # efficiency
    "yac_per_reception",     # what he does after the catch
    "epa_per_target",        # overall value per target
]

wr_rates[features].describe()

,targets_per_game,air_yards_per_target,catch_rate,yards_per_target,yac_per_reception,epa_per_target
count,67.000000,67.000000,67.000000,67.000000,67.000000,67.000000
mean,6.664179,11.259269,0.652903,8.527181,4.019898,0.344890
std,2.118440,3.386133,0.117516,2.574225,2.026652,0.407910
min,3.750000,3.777778,0.352941,3.764706,0.400000,-0.559636
25%,5.000000,9.562500,0.581989,6.567506,2.721111,0.055384
50%,6.333333,11.461538,0.650000,8.200000,3.454545,0.311153
75%,7.666667,13.524324,0.750000,10.312901,4.872995,0.603955
max,12.000000,18.400000,0.913043,16.000000,10.133333,1.399201


-- Standardize metrics with z-scores
-- Check data

In [13]:
z = (wr_rates[features] - wr_rates[features].mean()) / wr_rates[features].std()
z.index = wr_rates["player_display_name"].values
print(z)

                  targets_per_game  air_yards_per_target  catch_rate  \
Keenan Allen              0.001174             -1.494114   -0.024702   
Davante Adams             1.338636              1.617649   -0.629322   
Mike Evans               -0.628220             -0.095026    0.826246   
Stefon Diggs              0.512557             -0.657686   -0.614875   
Kalif Raymond            -0.431534             -1.655899    2.213661   
...                            ...                   ...         ...   
Antonio Williams         -0.903580              0.094408   -0.181455   
Malachi Fields           -1.375625              2.108816   -1.017474   
Ted Hurst III            -1.257614              1.787653   -1.301123   
Carnell Tate              0.630568              0.172610    0.294404   
KC Concepcion             0.158523             -1.500443   -0.389394   

                  yards_per_target  yac_per_reception  epa_per_target  
Keenan Allen             -1.370191          -0.882800       -1.

-- Define comparison function

In [14]:
def most_similar(name, n=5):
    distances = ((z - z.loc[name]) ** 2).sum(axis=1) ** 0.5
    return distances.drop(name).sort_values().head(n)

# pick any name from the list
wr_rates["player_display_name"].tolist()

['Keenan Allen',
 'Davante Adams',
 'Mike Evans',
 'Stefon Diggs',
 'Kalif Raymond',
 'Mack Hollins',
 'Chris Godwin Jr.',
 'Courtland Sutton',
 'DJ Moore',
 'Jakobi Meyers',
 'DK Metcalf',
 'Terry McLaurin',
 'Justin Jefferson',
 'CeeDee Lamb',
 'Tee Higgins',
 'Rashod Bateman',
 'Nico Collins',
 'Jaylen Waddle',
 "Ja'Marr Chase",
 'DeVonta Smith',
 'Amon-Ra St. Brown',
 'Drake London',
 'Chris Olave',
 'Jameson Williams',
 'George Pickens',
 'Khalil Shakir',
 'Garrett Wilson',
 'Romeo Doubs',
 "Wan'Dale Robinson",
 'Christian Watson',
 'Dontayvion Wicks',
 'Jaxon Smith-Njigba',
 'Quentin Johnston',
 'Michael Wilson',
 'Tre Tucker',
 'Parker Washington',
 'Xavier Hutchinson',
 'DeMario Douglas',
 'Jordan Addison',
 'Josh Downs',
 'Zay Flowers',
 'Rashee Rice',
 'Puka Nacua',
 'Malik Nabers',
 'Ryan Flournoy',
 'Devaughn Vele',
 'Jalen Coker',
 'Roman Wilson',
 'Malik Washington',
 'Adonai Mitchell',
 'Xavier Worthy',
 'Keon Coleman',
 'Ladd McConkey',
 'Rome Odunze',
 'Tetairoa McMill

TESTER
TESTER

In [15]:
most_similar("Jaxon Smith-Njigba", n=10)

CeeDee Lamb          0.633131
Chris Olave          1.296001
Nico Collins         1.832341
Tee Higgins          1.897090
Puka Nacua           2.379536
Jalen Coker          2.484231
Amon-Ra St. Brown    2.516530
Christian Watson     2.684186
Garrett Wilson       2.789791
Devaughn Vele        2.888267
dtype: float64

In [16]:
def weekly_view(name):
    cols = ["week", "targets", "receptions", "receiving_yards", "receiving_air_yards", "receiving_epa"]
    return wr[wr["player_display_name"] == name][cols].sort_values("week")

weekly_view("Davante Adams")

,week,targets,receptions,receiving_yards,receiving_air_yards,receiving_epa
23,1,6,3,26,73,-1.704729
1148,2,10,8,195,198,11.752949
2255,3,13,7,137,227,5.048632
3354,4,9,4,32,138,-0.176210
